# Large-Scale-OR 101: Single-Call Benchmark

This notebook runs the **Large-Scale-OR 101** single-call baseline for **Gemini 3.5 Flash**, with GPT-5.2 and Gemini 3.1 Pro retained as reproducibility options. Each problem receives exactly one tool-free API call, with the complete contents of that row's `Dataset_address` files included in the prompt.

The run records total tokens, input/output/reasoning token details, API calls, estimated API cost, response latency, raw response, extracted formulation, and generated Gurobi code. It does not read or execute any Air-NRM notebook.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'run_singlecall_cost_benchmark.py').is_file():
    candidate = NOTEBOOK_DIR / 'Test_Dataset' / 'Large-scale-or'
    if (candidate / 'run_singlecall_cost_benchmark.py').is_file():
        NOTEBOOK_DIR = candidate
    else:
        raise FileNotFoundError('Run this notebook from its own folder or the repository root.')
REPO_ROOT = NOTEBOOK_DIR.parents[1]
RUNNER = NOTEBOOK_DIR / 'run_singlecall_cost_benchmark.py'
print('Repository:', REPO_ROOT)
print('Runner:', RUNNER)

## 1. Choose the run

Use one model per run. Leave `ROWS = None` and `LIMIT = None` for the full 101-instance benchmark. For a cheap smoke test, use `ROWS = "0-2"`.

In [ ]:
MODEL = 'gemini-3.5-flash' # 'gemini-3.5-flash', 'gemini-3.1-pro', or 'gpt-5.2'
ROWS = None                # e.g. '0-2'; None selects all rows
LIMIT = None               # e.g. 3; None applies no additional limit
REPEAT_INDEX = 0
SOLVER_TIMEOUT_SECONDS = 300
MAX_RUN_COST_USD = 50.0
MAX_INSTANCE_COST_USD = 0.40 if MODEL.startswith('gemini-') else None
USE_PROXY = False
PROXY_URL = 'http://127.0.0.1:7897'
GENERATE_REPORT = True

if MODEL not in {'gpt-5.2', 'gemini-3.1-pro', 'gemini-3.5-flash'}:
    raise ValueError("MODEL must be 'gpt-5.2', 'gemini-3.1-pro', or 'gemini-3.5-flash'")

In [ ]:
def build_command(*, dry_run=False):
    command = [
        sys.executable, str(RUNNER), MODEL,
        '--repeat-index', str(REPEAT_INDEX),
        '--timeout', str(SOLVER_TIMEOUT_SECONDS),
        '--max-run-cost-usd', str(MAX_RUN_COST_USD),
        '--proxy-url', PROXY_URL,
        '--if-proxy' if USE_PROXY else '--no-proxy',
    ]
    if ROWS:
        command.extend(['--rows', ROWS])
    if LIMIT is not None:
        command.extend(['--limit', str(LIMIT)])
    if MAX_INSTANCE_COST_USD is not None:
        command.extend(['--max-instance-cost-usd', str(MAX_INSTANCE_COST_USD)])
    if not GENERATE_REPORT:
        command.append('--no-report')
    if dry_run:
        command.append('--dry-run')
    return command

print('Validating the model configuration, all 101 rows, and every Dataset_address file...')
subprocess.run(build_command(dry_run=True), cwd=REPO_ROOT, check=True)

## 2. Run the API benchmark

The key is entered without echo and is kept only in this notebook process. If the corresponding environment variable is already set, the prompt is skipped. Running the cell below starts paid API calls.

In [ ]:
key_variable = 'OPENAI_API_KEY' if MODEL == 'gpt-5.2' else 'GOOGLE_API_KEY'
if not os.getenv(key_variable):
    entered_key = getpass.getpass(f'Enter {key_variable} (input hidden): ').strip()
    if len(entered_key) < 20:
        raise ValueError(f'{key_variable} looks empty or incomplete')
    os.environ[key_variable] = entered_key

print(f'Starting {MODEL} on the selected Large-Scale-OR instances...')
subprocess.run(build_command(dry_run=False), cwd=REPO_ROOT, check=True)

## 3. Run the second model

Change `MODEL` in Section 1 to the other model, rerun Sections 1–2, and keep the same `ROWS`, `LIMIT`, timeout, and repeat index. Each model receives its own timestamped run directory under `runs/`.